In [1]:
import numpy as np
import torch
from torch import nn, autograd as ag
import matplotlib.pyplot as plt
from copy import deepcopy
import random

from tqdm import tqdm
import timm
from pytorch_datasets.ISIC2018 import ISIC2018
from pytorch_datasets.BreakHis import BreakHis

from easyfsl.samplers import TaskSampler
from easyfsl.methods.utils import evaluate
from torch.utils.data import DataLoader

In [2]:
seed = 0
plot = True
innerstepsize = 0.02 # stepsize in inner SGD
innerepochs = 1 # number of epochs of each inner SGD
outerstepsize0 = 0.1 # stepsize of outer optimization, i.e., meta-optimization
niterations = 30000 # number of outer updates; each iteration we sample one task and update on it

rng = np.random.RandomState(seed)
random_seed = 0
np.random.seed(random_seed)
torch.manual_seed(random_seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
n_way   = 2
n_shot  = 3
n_query = 1

n_episodes          = 40000
n_tasks_per_epoch   = 1
n_validation_tasks  = 100

DEVICE = "cuda"
n_workers = 4

datasets  = ['ISIC2018', 'BreakHis']
dataset   = 0

if dataset == 0:
  root  = 'E:/Datasets/ISIC2018'
  num_classes = 7
  train_set = ISIC2018(root=root, split='train', training=True)
  val_set   = ISIC2018(root=root, split='validation', training=False)

if dataset == 1:
  root          = 'E:/Datasets/BreaKHis_v1'
  magnification = 100
  num_classes = 8
  train_set = BreakHis(root=root, magnification=magnification, split='train', training=True)
  val_set   = BreakHis(root=root, magnification=magnification, split='validation', training=False)


# train_sampler = TaskSampler(
#     train_set, n_way=n_way, n_shot=n_shot, n_query=n_query, n_tasks=n_tasks_per_epoch
# )
# val_sampler = TaskSampler(
#     val_set, n_way=n_way, n_shot=n_shot, n_query=n_query, n_tasks=n_validation_tasks
# )

# # Finally, the DataLoader. We customize the collate_fn so that batches are delivered
# # in the shape: (support_images, support_labels, query_images, query_labels, class_ids)
# train_loader = DataLoader(
#     train_set,
#     batch_sampler=train_sampler,
#     num_workers=n_workers,
#     pin_memory=True,
#     collate_fn=train_sampler.episodic_collate_fn,
# )
# val_loader = DataLoader(
#     val_set,
#     batch_sampler=val_sampler,
#     num_workers=n_workers,
#     pin_memory=True,
#     collate_fn=val_sampler.episodic_collate_fn,
# )

In [4]:
def accuracy(predictions, targets):
    predictions = predictions.argmax(dim=1).view(targets.shape)
    return (predictions == targets).sum().float() / targets.size(0)

In [5]:
def fast_adapt(batch, learner, adapt_opt, loss, adaptation_steps, shots, ways, batch_size, device):
    data, labels, *_ = batch
    data, labels = data.to(device), labels.to(device)

    # Separate data into adaptation/evalutation sets
    adaptation_indices = np.zeros(data.size(0), dtype=bool)
    adaptation_indices[np.arange(shots*ways) * 2] = True
    evaluation_indices = torch.from_numpy(~adaptation_indices)
    adaptation_indices = torch.from_numpy(adaptation_indices)
    adaptation_data, adaptation_labels = data[adaptation_indices], labels[adaptation_indices]
    evaluation_data, evaluation_labels = data[evaluation_indices], labels[evaluation_indices]

    # Adapt the model
    for step in range(adaptation_steps):
        idx = torch.randint(
            adaptation_data.size(0),
            size=(batch_size, )
        )
        adapt_X = adaptation_data[idx]
        adapt_y = adaptation_labels[idx]
        adapt_opt.zero_grad()
        error = loss(learner(adapt_X), adapt_y)
        error.backward()
        adapt_opt.step()

    # Evaluate the adapted model
    predictions = learner(evaluation_data)
    valid_error = loss(predictions, evaluation_labels)
    valid_error /= len(evaluation_data)
    valid_accuracy = accuracy(predictions, evaluation_labels)
    return valid_error, valid_accuracy

In [6]:
import copy


def main(
        model,
        experiment='dev',
        problem='mini-imagenet',
        ways=5,
        train_shots=15,
        test_shots=5,
        meta_lr=1.0,
        meta_lr_final=1.0,
        meta_bsz=5,
        fast_lr=0.001,
        train_bsz=10,
        test_bsz=15,
        train_steps=8,
        test_steps=50,
        iterations=100,
        test_interval=5,
        save='',
        cuda=1,
        seed=42,
):
    cuda = bool(cuda)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    device = torch.device('cpu')
    if cuda and torch.cuda.device_count():
        device = torch.device('cuda')

    # train_tasks, valid_tasks, test_tasks = get_tasksets(
    #     train_samples=2*train_shots,
    #     train_ways=ways,
    #     test_samples=2*test_shots,
    #     test_ways=ways,
    #     root='~/data',
    # )
    # train_tasks = iter(train_loader)
    # test_tasks = iter(test_loader)
    # train_tasks, test_tasks = train_iter.next(), test_iter.next()

    train_sampler = TaskSampler(
    train_set, n_way=ways, n_shot=2*train_shots, n_query=1, n_tasks=meta_bsz)
    val_sampler = TaskSampler(
    val_set, n_way=ways, n_shot=2*test_shots, n_query=1, n_tasks=1)

    train_loader = DataLoader(
    train_set,
    batch_sampler=train_sampler,
    num_workers=n_workers,
    persistent_workers=True,
    pin_memory=True,
    collate_fn=train_sampler.episodic_collate_fn,)
    
    val_loader = DataLoader(
    val_set,
    batch_sampler=val_sampler,
    persistent_workers=True,
    num_workers=n_workers,
    pin_memory=True,
    collate_fn=val_sampler.episodic_collate_fn,)

    # Create model
    model.to(device)
    opt = torch.optim.SGD(model.parameters(), meta_lr)
    adapt_opt = torch.optim.Adam(model.parameters(), lr=fast_lr, betas=(0, 0.999))
    adapt_opt_state = adapt_opt.state_dict()
    loss = torch.nn.CrossEntropyLoss(reduction='mean')

    train_inner_errors = []
    train_inner_accuracies = []
    valid_inner_errors = []
    valid_inner_accuracies = []
    test_inner_errors = []
    test_inner_accuracies = []

    with tqdm(range(iterations)) as train_tqdm:
        for iteration in train_tqdm:
            opt.zero_grad()
            meta_train_error = 0.0
            meta_train_accuracy = 0.0
            meta_valid_error = 0.0
            meta_valid_accuracy = 0.0
            meta_test_error = 0.0
            meta_test_accuracy = 0.0

            # anneal meta-lr
            frac_done = float(iteration) / iterations
            new_lr = frac_done * meta_lr_final + (1 - frac_done) * meta_lr
            for pg in opt.param_groups:
                pg['lr'] = new_lr

            # zero-grad the parameters
            for p in model.parameters():
                p.grad = torch.zeros_like(p.data)

            for task in train_loader:
                # Compute meta-training loss
                learner = copy.deepcopy(model)
                adapt_opt = torch.optim.Adam(
                    learner.parameters(),
                    lr=fast_lr,
                    betas=(0, 0.999)
                )
                adapt_opt.load_state_dict(adapt_opt_state)
                batch = task
                evaluation_error, evaluation_accuracy = fast_adapt(batch,
                                                                learner,
                                                                adapt_opt,
                                                                loss,
                                                                train_steps,
                                                                train_shots,
                                                                ways,
                                                                train_bsz,
                                                                device)
                adapt_opt_state = adapt_opt.state_dict()
                for p, l in zip(model.parameters(), learner.parameters()):
                    p.grad.data.add_(-1.0, l.data)

                meta_train_error += evaluation_error.item()
                meta_train_accuracy += evaluation_accuracy.item()

                if iteration % test_interval == 0:
                    # Compute meta-validation loss
                    # learner = copy.deepcopy(model)
                    # adapt_opt = torch.optim.Adam(
                    #     learner.parameters(),
                    #     lr=fast_lr,
                    #     betas=(0, 0.999)
                    # )
                    # adapt_opt.load_state_dict(adapt_opt_state)
                    # batch = valid_tasks.sample()
                    # evaluation_error, evaluation_accuracy = fast_adapt(batch,
                    #                                                    learner,
                    #                                                    adapt_opt,
                    #                                                    loss,
                    #                                                    test_steps,
                    #                                                    test_shots,
                    #                                                    ways,
                    #                                                    test_bsz,
                    #                                                    device)
                    # meta_valid_error += evaluation_error.item()
                    # meta_valid_accuracy += evaluation_accuracy.item()

                    # Compute meta-testing loss
                    for test_task in val_loader:
                        learner = copy.deepcopy(model)
                        adapt_opt = torch.optim.Adam(
                            learner.parameters(),
                            lr=fast_lr,
                            betas=(0, 0.999)
                        )
                        adapt_opt.load_state_dict(adapt_opt_state)
                        batch = test_task
                        evaluation_error, evaluation_accuracy = fast_adapt(batch,
                                                                        learner,
                                                                        adapt_opt,
                                                                        loss,
                                                                        test_steps,
                                                                        test_shots,
                                                                        ways,
                                                                        test_bsz,
                                                                        device)
                        meta_test_error += evaluation_error.item()
                        meta_test_accuracy += evaluation_accuracy.item()

            # Print some metrics
            # print('\n')
            # print('Iteration', iteration)
            # print('Meta Train Error', meta_train_error / meta_bsz)
            # print('Meta Train Accuracy', meta_train_accuracy / meta_bsz)
            train_tqdm.set_postfix(
                train_error = meta_train_error / meta_bsz,
                train_acc   = meta_train_accuracy / meta_bsz
            )
            if iteration % test_interval == 0:
                # print('Meta Valid Error', meta_valid_error / meta_bsz)
                # print('Meta Valid Accuracy', meta_valid_accuracy / meta_bsz)
                # print('Meta Test Error', meta_test_error / meta_bsz)
                # print('Meta Test Accuracy', meta_test_accuracy / meta_bsz)
                train_tqdm.set_postfix(
                test_error = meta_test_error / meta_bsz,
                test_acc   = meta_test_accuracy / meta_bsz
                )

            # Track quantities
            train_inner_errors.append(meta_train_error / meta_bsz)
            train_inner_accuracies.append(meta_train_accuracy / meta_bsz)
            if iteration % test_interval == 0:
                valid_inner_errors.append(meta_valid_error / meta_bsz)
                valid_inner_accuracies.append(meta_valid_accuracy / meta_bsz)
                test_inner_errors.append(meta_test_error / meta_bsz)
                test_inner_accuracies.append(meta_test_accuracy / meta_bsz)

            # Average the accumulated gradients and optimize
            for p in model.parameters():
                p.grad.data.mul_(1.0 / meta_bsz).add_(p.data)
            opt.step()

    return test_inner_accuracies, test_inner_errors


In [9]:
model_name = 'vit_small_patch16_224'

vit = timm.create_model(model_name, pretrained=True, in_chans=3, num_classes=num_classes)
# vit.head = nn.Identity() # vit
# vit.head.fc = nn.Identity() # swin
# vit.fc = nn.Identity() # resnet50

# convolutional_network = resnet12()
model = vit.to(DEVICE)

criterion = nn.NLLLoss()

In [10]:
main(model=model, ways=n_way, train_shots=n_shot, test_shots=n_shot)

  0%|          | 0/100 [00:00<?, ?it/s]C:\Users\Main\AppData\Local\Temp\ipykernel_14636\769160556.py:120: UserWarning: This overload of add_ is deprecated:
	add_(Number alpha, Tensor other)
Consider using one of the following signatures instead:
	add_(Tensor other, *, Number alpha) (Triggered internally at  C:\cb\pytorch_1000000000000\work\torch\csrc\utils\python_arg_parser.cpp:1174.)
  p.grad.data.add_(-1.0, l.data)
100%|██████████| 100/100 [15:01<00:00,  9.02s/it, train_acc=0.667, train_error=0.162]


In [11]:
meta_lr=1.0
meta_lr_final=1.0
meta_bsz=5
fast_lr=0.001
n_validation_tasks  = 100
test_bsz=15
train_steps=8
test_steps=50
iterations=100000
test_interval=5
train_shots=15
test_shots=5
meta_lr=1.0
meta_lr_final=1.0
meta_bsz=5
fast_lr=0.001
train_bsz=10

val_sampler = TaskSampler(
val_set, n_way=n_way, n_shot=2*n_shot, n_query=1, n_tasks=n_validation_tasks)

val_loader = DataLoader(
val_set,
batch_sampler=val_sampler,
num_workers=n_workers,
persistent_workers=True,
pin_memory=True,
collate_fn=val_sampler.episodic_collate_fn,)

opt = torch.optim.SGD(model.parameters(), meta_lr)
adapt_opt = torch.optim.Adam(model.parameters(), lr=fast_lr, betas=(0, 0.999))
adapt_opt_state = adapt_opt.state_dict()
loss = torch.nn.CrossEntropyLoss(reduction='mean')

meta_test_error = 0.0
meta_test_accuracy = 0.0

with tqdm(
        enumerate(val_loader), total=len(val_loader), desc="Training"
    ) as tqdm_val:
    for test_task in tqdm_val:
        learner = copy.deepcopy(model)
        adapt_opt = torch.optim.Adam(
            learner.parameters(),
            lr=fast_lr,
            betas=(0, 0.999)
        )
        adapt_opt.load_state_dict(adapt_opt_state)
        batch = test_task
        evaluation_error, evaluation_accuracy = fast_adapt(batch,
                                                        learner,
                                                        adapt_opt,
                                                        loss,
                                                        test_steps,
                                                        n_shot,
                                                        n_way,
                                                        test_bsz,
                                                        'cuda')
        meta_test_error += evaluation_error.item()
        meta_test_accuracy += evaluation_accuracy.item()


In [13]:
print('Meta Test Error', meta_test_error / 100)
print('Meta Test Accuracy', meta_test_accuracy / 100)

Meta Test Error 0.3169273254252039
Meta Test Accuracy 0.6066666790843009


In [9]:
# if __name__ == '__main__':
#     main()